# C3: Multimodal Early Warning Risk Prediction (EDA & Baseline)

**Component 3:** Multimodal Risk Prediction for Student Dropout Early Warning System  
**Author:** Thilina  
**Data Contract:** `contracts/v1/RiskAssessment.json` & `contracts/CONTRACT.md`

---

### Research Objectives
1. **Triangulate 3 Modalities:** Combine **Affective signals** (C1), **Behavioral velocity** (C2), and **Academic performance** (LMS).
2. **Predict Composite Risk:** Generate a weekly `risk_score` ($0.0 - 1.0$) and categorize into a `risk_tier` (`low`, `medium`, `high`, `critical`).
3. **Explainability via SHAP:** Calculate `modality_contributions` (Affective vs. Behavioral vs. Academic percentage) and `top_drivers`.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path
sys.path.append(os.path.abspath(".."))
from src.model import MultimodalRiskModel, ALL_FEATURE_COLS
from src.generate_multimodal_dataset import generate_multimodal_data

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

## 1. Load Multimodal Dataset

In [ ]:
data_path = "../data/multimodal_risk_training_data.csv"
if not os.path.exists(data_path):
    print("Generating multimodal dataset...")
    df = generate_multimodal_data(num_students=400, num_weeks=12)
    df.to_csv(data_path, index=False)
else:
    df = pd.read_csv(data_path)

print(f"Loaded {len(df)} records across {df['pid'].nunique()} unique students.")
df.head()

## 2. Exploratory Data Analysis & Multimodal Distributions

In [ ]:
# Risk Tier Distribution
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="risk_tier", order=["low", "medium", "high", "critical"], palette="viridis")
plt.title("Student Weekly Risk Tier Distribution")
plt.xlabel("Risk Tier")
plt.ylabel("Count")
plt.show()

In [ ]:
# Multimodal Correlation Heatmap
corr_cols = [
    "cumulative_grade", "quiz_avg_score", "assignment_submission_delay_days",
    "weekly_clicks", "video_watch_minutes", "active_days_count",
    "exam_anxiety", "conceptual_confusion", "academic_helplessness",
    "actual_composite_risk", "actual_dropout"
]

plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Cross-Modality Correlation Heatmap")
plt.show()

## 3. Train Multimodal Early Warning Model (with SHAP Explainability)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Student-level split to prevent data leakage
pids = df["pid"].unique()
train_pids, test_pids = train_test_split(pids, test_size=0.20, random_state=42)

train_df = df[df["pid"].isin(train_pids)]
test_df = df[df["pid"].isin(test_pids)]

X_train, y_train = train_df[ALL_FEATURE_COLS], train_df["actual_dropout"].values
X_test, y_test = test_df[ALL_FEATURE_COLS], test_df["actual_dropout"].values

model = MultimodalRiskModel(n_estimators=120, max_depth=5, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict_risk(X_test)
pred_scores = [p["risk_score"] for p in predictions]
print(f"Test ROC-AUC: {roc_auc_score(y_test, pred_scores):.4f}")

## 4. Contract-Compliant Explainability Inspection

In [ ]:
# Inspect first 3 test students
for i in range(3):
    p = predictions[i]
    print(f"\n--- Student {test_df.iloc[i]['pid']} (Week {test_df.iloc[i]['week_index']}) ---")
    print(f"Predicted Risk Score: {p['risk_score']} ({p['risk_tier'].upper()})")
    print(f"Modality Contributions: {p['modality_contributions']}")
    print(f"Top Risk Drivers: {p['top_drivers']}")